<a href="https://colab.research.google.com/github/NoisyArchie/VisionArtificial/blob/develop/VisionArtificial_Fase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 2 — Desarrollo ML: preprocesamiento y fine-tuning de YOLO
**Proyecto:** Implementación de un sistema de visión artificial basado en arquitecturas de detección de una sola etapa para la gestión de aforos en tiempo real
**Autor:** Ángel de Jesús Sánchez Jaramillo · **Asesor:** Ernesto Liñán García · UAdeC, Facultad de Sistemas

| Sección | Contenido |
|---|---|
| 1 | Configuración del entorno (GPU, Drive, librerías, parámetros) |
| 2 | Módulos del proyecto (`src/`) |
| 3 | Preparación de datos: CrowdHuman → formato YOLO |
| 4 | Preprocesamiento de video con OpenCV |
| 5 | Línea base: YOLO preentrenado en COCO |
| 6 | Fine-tuning en CrowdHuman |
| 7 | Evaluación y comparación |
| 8 | Prueba de rendimiento (FPS) |
| 9 | Siguientes pasos (Fase 3) |

> **Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

## 1. Configuración del entorno

In [1]:
!nvidia-smi

Tue Oct  6 02:35:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, sys
from pathlib import Path

# ---- Rutas: todo lo persistente vive en Drive ----
BASE = Path('/content/drive/MyDrive/VisionArtificial')
for d in ['datasets/crowdhuman/raw', 'datasets/mot20', 'configs', 'src', 'runs', 'resultados', 'videos_prueba']:
    (BASE / d).mkdir(parents=True, exist_ok=True)

# ---- Rutas locales (rápidas, pero se borran al desconectar) ----
LOCAL = Path('/content/datos')
RAW_LOCAL = LOCAL / 'crowdhuman_raw'
YOLO_LOCAL = LOCAL / 'crowdhuman_yolo'
RAW_LOCAL.mkdir(parents=True, exist_ok=True)

sys.path.append(str(BASE / 'src'))
print('Carpeta del proyecto:', BASE)

Carpeta del proyecto: /content/drive/MyDrive/VisionArtificial


In [4]:
!pip install -q ultralytics gdown
import ultralytics, torch
ultralytics.checks()
print('CUDA disponible:', torch.cuda.is_available())

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.5/112.6 GB disk)
CUDA disponible: True


### Parámetros del experimento
Cambia aquí y no dentro del código. Cada combinación genera una carpeta distinta en `runs/`, así puedes comparar experimentos en el informe.

In [5]:
CONFIG = {
    # Datos
    'partes_train': ['CrowdHuman_train01'],   # agrega train02/train03 si tienes tiempo y espacio
    'tipo_caja': 'vbox',                      # 'vbox' visible | 'fbox' completo | 'hbox' cabeza
    'min_lado_px': 4,
    # Modelo y entrenamiento
    'modelo': 'yolov8s.pt',                   # n = más rápido, s = buen balance en T4, m = más preciso
    'epocas': 50,
    'imgsz': 640,
    'batch': 16,                              # bájalo a 8 si sale 'CUDA out of memory'
    'paciencia': 15,                          # early stopping
    'guardar_cada': 5,                        # checkpoint cada n épocas (por si Colab se desconecta)
}
NOMBRE_EXP = f"{Path(CONFIG['modelo']).stem}_{CONFIG['tipo_caja']}_{len(CONFIG['partes_train'])}parte"
ZIP_YOLO_DRIVE = BASE / 'datasets' / 'crowdhuman' / f"crowdhuman_yolo_{CONFIG['tipo_caja']}_{len(CONFIG['partes_train'])}parte.zip"
YAML = BASE / 'configs' / f"crowdhuman_{CONFIG['tipo_caja']}.yaml"
print('Experimento:', NOMBRE_EXP)

Experimento: yolov8s_vbox_1parte


## 2. Módulos del proyecto
Estas celdas escriben los archivos `.py` en `VisionArtificial/src/` de tu Drive. Solo necesitas volver a ejecutarlas si modificas el código.

In [ ]:
%%writefile /content/drive/MyDrive/VisionArtificial/src/conversion_datos.py
"""
conversion_datos.py
Conversión de las anotaciones de CrowdHuman (.odgt) al formato que usa Ultralytics YOLO.

Formato .odgt: cada línea es un JSON con las anotaciones de una imagen:
{"ID": "273271,c9db000d5146c15",
 "gtboxes": [{"tag": "person", "fbox": [x, y, w, h], "vbox": [...], "hbox": [...],
              "extra": {"ignore": 0}, "head_attr": {"ignore": 0}}, ...]}

Formato YOLO: un .txt por imagen, una línea por objeto:
<clase> <x_centro> <y_centro> <ancho> <alto>   (valores normalizados entre 0 y 1)
"""

import json
import random
import shutil
from collections import Counter
from pathlib import Path

import cv2
from PIL import Image

TIPOS_CAJA = {
    "vbox": "cuerpo visible",
    "fbox": "cuerpo completo",
    "hbox": "cabeza",
}


def leer_odgt(ruta):
    """Lee un archivo .odgt y regresa un diccionario por imagen (una línea = un JSON)."""
    with open(ruta, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if linea:
                yield json.loads(linea)


def es_ignorada(gt, tipo_caja):
    """True si la anotación no debe usarse (no es persona o está marcada como 'ignore')."""
    if gt.get("tag") != "person":  # 'mask' = maniquíes, reflejos, carteles, etc.
        return True
    if gt.get("extra", {}).get("ignore", 0) == 1:
        return True
    if tipo_caja == "hbox" and gt.get("head_attr", {}).get("ignore", 0) == 1:
        return True
    return False


def caja_a_yolo(caja, ancho_img, alto_img):
    """Convierte [x, y, w, h] en píxeles a (xc, yc, w, h) normalizados.
    Recorta la caja a los bordes de la imagen (fbox puede salirse de la imagen)."""
    x, y, w, h = caja
    x1, y1 = max(0.0, x), max(0.0, y)
    x2, y2 = min(float(ancho_img), x + w), min(float(alto_img), y + h)
    w_rec, h_rec = x2 - x1, y2 - y1
    if w_rec <= 0 or h_rec <= 0:
        return None
    return (
        (x1 + w_rec / 2) / ancho_img,
        (y1 + h_rec / 2) / alto_img,
        w_rec / ancho_img,
        h_rec / alto_img,
        w_rec,
        h_rec,
    )


def indexar_imagenes(carpetas):
    """Busca todas las .jpg dentro de las carpetas y regresa {ID: ruta}."""
    indice = {}
    for carpeta in carpetas:
        for ruta in Path(carpeta).rglob("*.jpg"):
            indice[ruta.stem] = ruta
    return indice


def analizar_odgt(ruta_odgt):
    """Estadísticas rápidas del dataset para el reporte."""
    personas_por_imagen = []
    etiquetas = Counter()
    ignoradas = 0
    for registro in leer_odgt(ruta_odgt):
        n = 0
        for gt in registro.get("gtboxes", []):
            etiquetas[gt.get("tag")] += 1
            if gt.get("tag") == "person" and gt.get("extra", {}).get("ignore", 0) == 0:
                n += 1
            else:
                ignoradas += 1
        personas_por_imagen.append(n)
    total = len(personas_por_imagen)
    return {
        "imagenes": total,
        "personas": sum(personas_por_imagen),
        "promedio_por_imagen": sum(personas_por_imagen) / total if total else 0,
        "maximo_por_imagen": max(personas_por_imagen) if total else 0,
        "anotaciones_ignoradas": ignoradas,
        "etiquetas": dict(etiquetas),
        "personas_por_imagen": personas_por_imagen,
    }


def convertir_split(ruta_odgt, carpetas_imagenes, destino, split,
                    tipo_caja="vbox", min_lado_px=4, mover=False, limite=None):
    """
    Convierte un split (train o val) de CrowdHuman a formato YOLO.

    ruta_odgt         : annotation_train.odgt o annotation_val.odgt
    carpetas_imagenes : lista de carpetas donde están las .jpg descomprimidas
    destino           : carpeta raíz del dataset YOLO (se crean images/<split> y labels/<split>)
    tipo_caja         : 'vbox' (visible), 'fbox' (completo) o 'hbox' (cabeza)
    min_lado_px       : descarta cajas con ancho o alto menor a esto (ruido)
    mover             : True mueve las imágenes en lugar de copiarlas (ahorra disco en Colab)
    limite            : número máximo de imágenes (útil para pruebas rápidas)
    """
    if tipo_caja not in TIPOS_CAJA:
        raise ValueError(f"tipo_caja debe ser uno de {list(TIPOS_CAJA)}")

    destino = Path(destino)
    dir_img = destino / "images" / split
    dir_lbl = destino / "labels" / split
    dir_img.mkdir(parents=True, exist_ok=True)
    dir_lbl.mkdir(parents=True, exist_ok=True)

    indice = indexar_imagenes(carpetas_imagenes)
    stats = Counter()

    for registro in leer_odgt(ruta_odgt):
        if limite is not None and stats["imagenes"] >= limite:
            break
        id_img = registro["ID"]
        ruta_img = indice.get(id_img)
        if ruta_img is None:
            stats["sin_imagen"] += 1  # pasa si solo descomprimiste una parte del train
            continue

        with Image.open(ruta_img) as im:
            ancho, alto = im.size

        lineas = []
        for gt in registro.get("gtboxes", []):
            if es_ignorada(gt, tipo_caja) or tipo_caja not in gt:
                stats["cajas_ignoradas"] += 1
                continue
            res = caja_a_yolo(gt[tipo_caja], ancho, alto)
            if res is None or res[4] < min_lado_px or res[5] < min_lado_px:
                stats["cajas_pequenas"] += 1
                continue
            xc, yc, w, h = res[:4]
            lineas.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        (dir_lbl / f"{id_img}.txt").write_text("\n".join(lineas))
        destino_img = dir_img / f"{id_img}.jpg"
        if not destino_img.exists():
            if mover:
                shutil.move(str(ruta_img), destino_img)
            else:
                shutil.copy2(ruta_img, destino_img)

        stats["imagenes"] += 1
        stats["cajas"] += len(lineas)
        if not lineas:
            stats["imagenes_sin_personas"] += 1

    return dict(stats)


def generar_yaml(destino, tipo_caja="vbox", ruta_yaml=None):
    """Crea el archivo .yaml que Ultralytics necesita para entrenar."""
    destino = Path(destino)
    ruta_yaml = Path(ruta_yaml) if ruta_yaml else destino / "crowdhuman.yaml"
    contenido = (
        f"# CrowdHuman en formato YOLO - caja: {tipo_caja} ({TIPOS_CAJA[tipo_caja]})\n"
        f"path: {destino}\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: persona\n"
    )
    ruta_yaml.parent.mkdir(parents=True, exist_ok=True)
    ruta_yaml.write_text(contenido)
    return ruta_yaml


def dibujar_etiquetas(ruta_img, ruta_txt, color=(0, 255, 0), grosor=2):
    """Dibuja las cajas YOLO sobre la imagen para verificar la conversión. Regresa RGB."""
    img = cv2.imread(str(ruta_img))
    alto, ancho = img.shape[:2]
    texto = Path(ruta_txt).read_text().strip()
    n = 0
    for linea in texto.splitlines() if texto else []:
        _, xc, yc, w, h = map(float, linea.split())
        x1 = int((xc - w / 2) * ancho)
        y1 = int((yc - h / 2) * alto)
        x2 = int((xc + w / 2) * ancho)
        y2 = int((yc + h / 2) * alto)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, grosor)
        n += 1
    cv2.putText(img, f"personas: {n}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def muestras_aleatorias(destino, split="val", n=4, semilla=0):
    """Regresa n pares (imagen, etiqueta) al azar para inspección visual."""
    destino = Path(destino)
    imagenes = sorted((destino / "images" / split).glob("*.jpg"))
    random.Random(semilla).shuffle(imagenes)
    return [(p, destino / "labels" / split / f"{p.stem}.txt") for p in imagenes[:n]]

In [ ]:
%%writefile /content/drive/MyDrive/VisionArtificial/src/preprocesamiento.py
"""
preprocesamiento.py
Pipeline de preprocesamiento de video con OpenCV.

Nota de diseño: Ultralytics YOLO ya aplica internamente letterbox, BGR->RGB y
normalización 0-1. Por eso el Preprocesador NO repite esos pasos antes de la
inferencia; se encarga de lo que el modelo no hace solo:
  - corrección de iluminación (CLAHE y gamma automática para escenas oscuras)
  - región de interés (ROI) para ignorar zonas del cuadro que no importan
  - reducción de resolución para ganar FPS
Las funciones letterbox() y normalizar() se incluyen para documentar y
visualizar lo que sucede dentro del modelo (sirve para el informe técnico).
"""

import time

import cv2
import numpy as np


# ---------------------------------------------------------------- utilidades
def letterbox(img, tam=640, color=(114, 114, 114)):
    """Redimensiona sin deformar y rellena con bordes hasta tam x tam.
    Regresa (imagen, escala, (relleno_x, relleno_y))."""
    alto, ancho = img.shape[:2]
    r = min(tam / alto, tam / ancho)
    nuevo_ancho, nuevo_alto = int(round(ancho * r)), int(round(alto * r))
    redim = cv2.resize(img, (nuevo_ancho, nuevo_alto), interpolation=cv2.INTER_LINEAR)
    arriba = (tam - nuevo_alto) // 2
    izquierda = (tam - nuevo_ancho) // 2
    salida = cv2.copyMakeBorder(
        redim, arriba, tam - nuevo_alto - arriba, izquierda, tam - nuevo_ancho - izquierda,
        cv2.BORDER_CONSTANT, value=color,
    )
    return salida, r, (izquierda, arriba)


def normalizar(img):
    """BGR uint8 (0-255) -> RGB float32 (0-1). Equivale a lo que hace YOLO por dentro."""
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0


def brillo_medio(img):
    """Brillo promedio (0-255) usando el canal de luminancia."""
    return float(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean())


def aplicar_clahe(img, clip=2.0, rejilla=8):
    """Ecualización adaptativa de histograma sobre el canal L (LAB).
    Mejora el contraste local sin alterar los colores."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(rejilla, rejilla))
    return cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)


def correccion_gamma(img, gamma):
    """Aplica salida = (entrada/255)^gamma * 255. gamma < 1 aclara, gamma > 1 oscurece."""
    tabla = (np.linspace(0, 1, 256) ** gamma * 255).clip(0, 255).astype(np.uint8)
    return cv2.LUT(img, tabla)


def gamma_automatica(img, objetivo=128, minimo=0.4):
    """Calcula la gamma que lleva el brillo medio hacia 'objetivo'. Solo aclara."""
    media = max(brillo_medio(img), 1.0)
    if media >= objetivo:
        return 1.0
    gamma = np.log(objetivo / 255) / np.log(media / 255)
    return float(np.clip(gamma, minimo, 1.0))


def construir_mascara_roi(forma, poligono):
    """Máscara binaria a partir de un polígono [(x, y), ...] en píxeles."""
    mascara = np.zeros(forma[:2], dtype=np.uint8)
    cv2.fillPoly(mascara, [np.array(poligono, dtype=np.int32)], 255)
    return mascara


def aplicar_roi(img, mascara):
    """Pone en negro todo lo que está fuera de la región de interés."""
    return cv2.bitwise_and(img, img, mask=mascara)


def redimensionar_max(img, lado_max):
    """Reduce la imagen para que su lado mayor no pase de lado_max (no agranda)."""
    alto, ancho = img.shape[:2]
    escala = lado_max / max(alto, ancho)
    if escala >= 1:
        return img, 1.0
    return cv2.resize(img, (int(ancho * escala), int(alto * escala)),
                      interpolation=cv2.INTER_AREA), escala


# ---------------------------------------------------------------- pipeline
class Preprocesador:
    """
    Pipeline configurable que se aplica a cada cuadro antes de pasarlo a YOLO.

    lado_max          : None o entero; reduce la resolución de entrada
    clahe             : 'auto' (solo en escenas oscuras), True (siempre) o False
    gamma             : True para aplicar gamma automática en escenas oscuras
    umbral_oscuridad  : brillo medio (0-255) por debajo del cual la escena se considera oscura
    roi               : None o lista de puntos [(x, y), ...] en coordenadas del cuadro ya redimensionado
    """

    def __init__(self, lado_max=None, clahe="auto", gamma=True, umbral_oscuridad=80,
                 clip=2.0, roi=None):
        self.lado_max = lado_max
        self.clahe = clahe
        self.gamma = gamma
        self.umbral_oscuridad = umbral_oscuridad
        self.clip = clip
        self.roi = roi
        self._mascara = None
        self.tiempos_ms = []
        self.ultima_info = {}

    def __call__(self, frame):
        t0 = time.perf_counter()
        info = {}

        if self.lado_max:
            frame, info["escala"] = redimensionar_max(frame, self.lado_max)

        brillo = brillo_medio(frame)
        oscura = brillo < self.umbral_oscuridad
        info["brillo"] = round(brillo, 1)
        info["oscura"] = oscura

        if self.gamma and oscura:
            g = gamma_automatica(frame)
            frame = correccion_gamma(frame, g)
            info["gamma"] = round(g, 3)

        if self.clahe is True or (self.clahe == "auto" and oscura):
            frame = aplicar_clahe(frame, clip=self.clip)
            info["clahe"] = True

        if self.roi is not None:
            if self._mascara is None or self._mascara.shape[:2] != frame.shape[:2]:
                self._mascara = construir_mascara_roi(frame.shape, self.roi)
            frame = aplicar_roi(frame, self._mascara)

        self.tiempos_ms.append((time.perf_counter() - t0) * 1000)
        self.ultima_info = info
        return frame

    def resumen_tiempos(self):
        if not self.tiempos_ms:
            return {}
        t = np.array(self.tiempos_ms)
        return {"cuadros": len(t), "promedio_ms": round(float(t.mean()), 2),
                "p95_ms": round(float(np.percentile(t, 95)), 2)}


# ---------------------------------------------------------------- video
def iterar_video(ruta, saltar=0):
    """Generador de cuadros de un video. saltar=n procesa 1 de cada n+1 cuadros."""
    cap = cv2.VideoCapture(str(ruta))
    if not cap.isOpened():
        raise IOError(f"No se pudo abrir el video: {ruta}")
    i = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if saltar == 0 or i % (saltar + 1) == 0:
                yield i, frame
            i += 1
    finally:
        cap.release()


def info_video(ruta):
    cap = cv2.VideoCapture(str(ruta))
    datos = {
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "cuadros": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "ancho": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "alto": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    cap.release()
    return datos


def simular_oscuridad(img, factor=0.35):
    """Oscurece una imagen para probar el pipeline sin necesitar video nocturno."""
    return (img.astype(np.float32) * factor).clip(0, 255).astype(np.uint8)

In [ ]:
import importlib
import conversion_datos as cd
import preprocesamiento as pp
importlib.reload(cd); importlib.reload(pp)
print('Módulos cargados')

## 3. Preparación de datos: CrowdHuman → YOLO

**Estrategia para no pelear con Colab:**
1. La primera vez se descarga CrowdHuman, se convierte y el resultado se guarda **comprimido en Drive**.
2. En sesiones siguientes solo se copia ese zip y se descomprime en `/content` (mucho más rápido que leer miles de imágenes desde Drive).

CrowdHuman completo pesa ~13 GB comprimido. Con `partes_train = ['CrowdHuman_train01']` + validación son ~5 GB, que caben en los 15 GB gratis de Drive y ya dan 5,000 imágenes de entrenamiento con ~23 personas cada una.

In [ ]:
YA_CONVERTIDO = ZIP_YOLO_DRIVE.exists()
if YA_CONVERTIDO:
    print('Dataset convertido encontrado en Drive. Copiando y descomprimiendo...')
    !cp "{ZIP_YOLO_DRIVE}" /content/datos/ds.zip
    !unzip -q -o /content/datos/ds.zip -d "{YOLO_LOCAL}" && rm /content/datos/ds.zip
    cd.generar_yaml(YOLO_LOCAL, CONFIG['tipo_caja'], YAML)
    print('Listo. Puedes saltar a la sección 4.')
else:
    print('No hay dataset convertido todavía: ejecuta las celdas 3.1 a 3.5.')

### 3.1 Descarga
Primero busca los archivos en `VisionArtificial/datasets/crowdhuman/raw/` de tu Drive. Si no están, intenta descargarlos de los enlaces oficiales de Google Drive con `gdown`.

**Si `gdown` falla** (pasa seguido por el límite de descargas de Google Drive): descarga los archivos manualmente desde https://www.crowdhuman.org/download.html, súbelos a `VisionArtificial/datasets/crowdhuman/raw/` y vuelve a ejecutar la celda.

In [ ]:
import shutil, gdown

ARCHIVOS = {  # IDs de los enlaces oficiales de CrowdHuman
    'CrowdHuman_train01.zip': '134QOvaatwKdy0iIeNqA_p-xkAhkV4F8Y',
    'CrowdHuman_train02.zip': '17evzPh7gc1JBNvnW1ENXLy5Kr4Q_Nnla',
    'CrowdHuman_train03.zip': '1tdp0UCgxrqy1B6p8LkR-Iy0aIJ8l4fJW',
    'CrowdHuman_val.zip':     '18jFI789CoHTppQ7vmRSFEdnGaSQZ4YzO',
    'annotation_train.odgt':  '1UUTea5mYqvlUObsC1Z8CFldHJAtLtMX3',
    'annotation_val.odgt':    '10WIRwu8ju8GRLuCkZ_vT6hnNxs5ptwoL',
}
necesarios = [f'{p}.zip' for p in CONFIG['partes_train']] + ['CrowdHuman_val.zip', 'annotation_train.odgt', 'annotation_val.odgt']
RAW_DRIVE = BASE / 'datasets' / 'crowdhuman' / 'raw'

for nombre in necesarios:
    destino = RAW_LOCAL / nombre
    if destino.exists():
        print('✓ ya existe', nombre); continue
    if (RAW_DRIVE / nombre).exists():
        print('→ copiando desde Drive', nombre)
        shutil.copy2(RAW_DRIVE / nombre, destino)
    else:
        print('↓ descargando', nombre)
        gdown.download(id=ARCHIVOS[nombre], output=str(destino), quiet=False)
    if not destino.exists():
        print(f'✗ No se pudo obtener {nombre}. Súbelo manualmente a {RAW_DRIVE}')

In [ ]:
# Descomprimir y borrar los zip para liberar disco
for zip_ in sorted(RAW_LOCAL.glob('*.zip')):
    carpeta = RAW_LOCAL / zip_.stem
    if not carpeta.exists():
        print('Descomprimiendo', zip_.name)
        !unzip -q "{zip_}" -d "{carpeta}"
    zip_.unlink()
!du -sh "{RAW_LOCAL}"/*

### 3.2 Análisis exploratorio
Estas estadísticas te sirven directamente para el capítulo de datos del informe.

In [ ]:
import matplotlib.pyplot as plt

estadisticas = {}
for split, archivo in [('train', 'annotation_train.odgt'), ('val', 'annotation_val.odgt')]:
    estadisticas[split] = cd.analizar_odgt(RAW_LOCAL / archivo)
    e = estadisticas[split]
    print(f"{split}: {e['imagenes']} imágenes | {e['personas']} personas | "
          f"{e['promedio_por_imagen']:.1f} por imagen (máx {e['maximo_por_imagen']}) | "
          f"ignoradas: {e['anotaciones_ignoradas']}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(estadisticas['train']['personas_por_imagen'], bins=60, color='steelblue')
ax.set_xlabel('Personas por imagen'); ax.set_ylabel('Número de imágenes')
ax.set_title('CrowdHuman (train completo): densidad de personas')
plt.tight_layout(); plt.savefig(BASE / 'resultados' / 'densidad_crowdhuman.png', dpi=150); plt.show()

### 3.3 Conversión a formato YOLO
Nota: el `annotation_train.odgt` incluye las 15,000 imágenes de train, pero solo se convierten las de las partes que descomprimiste (las demás aparecen como `sin_imagen`, es normal).

In [ ]:
carpetas_train = [RAW_LOCAL / p for p in CONFIG['partes_train']]
for split, odgt, carpetas in [('train', 'annotation_train.odgt', carpetas_train),
                              ('val', 'annotation_val.odgt', [RAW_LOCAL / 'CrowdHuman_val'])]:
    stats = cd.convertir_split(RAW_LOCAL / odgt, carpetas, YOLO_LOCAL, split,
                               tipo_caja=CONFIG['tipo_caja'], min_lado_px=CONFIG['min_lado_px'], mover=True)
    print(split, stats)

cd.generar_yaml(YOLO_LOCAL, CONFIG['tipo_caja'], YAML)
print(YAML.read_text())

### 3.4 Verificación visual
Siempre revisa que las cajas caigan donde deben antes de entrenar: un error aquí arruina todo el entrenamiento.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (img, lbl) in zip(axes.flat, cd.muestras_aleatorias(YOLO_LOCAL, 'val', 4, semilla=1)):
    ax.imshow(cd.dibujar_etiquetas(img, lbl)); ax.set_title(img.name, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.savefig(BASE / 'resultados' / f"verificacion_{CONFIG['tipo_caja']}.png", dpi=120); plt.show()

### 3.5 Guardar el dataset convertido en Drive
Se comprime sin compresión (`-0`) porque las imágenes ya son JPG; así es mucho más rápido.

In [ ]:
!cd "{YOLO_LOCAL}" && zip -q -r -0 /content/datos/ds.zip images labels
!cp /content/datos/ds.zip "{ZIP_YOLO_DRIVE}" && rm /content/datos/ds.zip
!rm -rf "{RAW_LOCAL}"
!ls -lh "{ZIP_YOLO_DRIVE}"

## 4. Preprocesamiento de video con OpenCV
YOLO ya hace internamente letterbox y normalización (se muestra abajo para documentarlo). El módulo propio se enfoca en **iluminación variable** (gamma + CLAHE solo cuando la escena es oscura) y **región de interés**.

In [ ]:
import cv2, numpy as np

img_ejemplo = cv2.imread(str(cd.muestras_aleatorias(YOLO_LOCAL, 'val', 1, semilla=3)[0][0]))
oscura = pp.simular_oscuridad(img_ejemplo, 0.3)
prep = pp.Preprocesador(clahe='auto', gamma=True, umbral_oscuridad=80)
corregida = prep(oscura)
lb, escala, relleno = pp.letterbox(img_ejemplo, 640)

titulos = ['Original', f'Oscura simulada (brillo {pp.brillo_medio(oscura):.0f})',
           f'Corregida {prep.ultima_info}', f'Letterbox 640 (escala {escala:.2f})']
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, im, t in zip(axes, [img_ejemplo, oscura, corregida, lb], titulos):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(t, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.savefig(BASE / 'resultados' / 'preprocesamiento.png', dpi=120); plt.show()

In [ ]:
# Costo del preprocesamiento por cuadro (debe ser pequeño frente a la inferencia)
prep = pp.Preprocesador(lado_max=1280, clahe='auto', gamma=True)
for _ in range(100):
    prep(oscura)
print(prep.resumen_tiempos())

## 5. Línea base: YOLO preentrenado en COCO (sin fine-tuning)
Mide qué tan bien detecta personas el modelo "de fábrica" en CrowdHuman. Es el punto de comparación que justifica el fine-tuning en el informe.

In [ ]:
from ultralytics import YOLO
import json

modelo_base = YOLO(CONFIG['modelo'])
try:
    # La clase 0 en COCO también es 'person', así que se filtra solo esa clase
    m_base = modelo_base.val(data=str(YAML), imgsz=CONFIG['imgsz'], batch=CONFIG['batch'],
                             classes=[0], project=str(BASE / 'runs'), name=f'{NOMBRE_EXP}_base', exist_ok=True)
    METRICAS_BASE = {'mAP50': m_base.box.map50, 'mAP50-95': m_base.box.map,
                     'precision': m_base.box.mp, 'recall': m_base.box.mr}
except Exception as e:
    print('No se pudo evaluar la línea base:', e); METRICAS_BASE = None
print(METRICAS_BASE)

## 6. Fine-tuning en CrowdHuman
- Si Colab se desconecta, vuelve a correr las secciones 1, 2 y la primera celda de la 3, y luego esta celda: detecta el `last.pt` en Drive y **continúa** donde se quedó.
- En una T4 con `yolov8s`, 5,000 imágenes y 640 px, calcula unos minutos por época. Si el tiempo de sesión no te alcanza, baja `epocas` o usa `yolov8n.pt`.

In [ ]:
DIR_EXP = BASE / 'runs' / NOMBRE_EXP
ULTIMO = DIR_EXP / 'weights' / 'last.pt'

if ULTIMO.exists():
    print('Continuando entrenamiento desde', ULTIMO)
    modelo = YOLO(str(ULTIMO))
    resultados = modelo.train(resume=True)
else:
    modelo = YOLO(CONFIG['modelo'])
    resultados = modelo.train(
        data=str(YAML), epochs=CONFIG['epocas'], imgsz=CONFIG['imgsz'], batch=CONFIG['batch'],
        patience=CONFIG['paciencia'], save_period=CONFIG['guardar_cada'],
        project=str(BASE / 'runs'), name=NOMBRE_EXP, exist_ok=True,
        single_cls=True, cache=False, workers=2, seed=42, plots=True,
    )

## 7. Evaluación y comparación

In [ ]:
import pandas as pd
from IPython.display import Image as IImage, display

MEJOR = DIR_EXP / 'weights' / 'best.pt'
modelo_ft = YOLO(str(MEJOR))
m_ft = modelo_ft.val(data=str(YAML), imgsz=CONFIG['imgsz'], batch=CONFIG['batch'],
                     project=str(BASE / 'runs'), name=f'{NOMBRE_EXP}_val', exist_ok=True)
METRICAS_FT = {'mAP50': m_ft.box.map50, 'mAP50-95': m_ft.box.map,
               'precision': m_ft.box.mp, 'recall': m_ft.box.mr}

tabla = pd.DataFrame({'COCO (base)': METRICAS_BASE or {}, 'Fine-tuning CrowdHuman': METRICAS_FT}).round(4)
display(tabla)

with open(BASE / 'resultados' / f'metricas_{NOMBRE_EXP}.json', 'w') as f:
    json.dump({'config': CONFIG, 'base': METRICAS_BASE, 'fine_tuning': METRICAS_FT}, f, indent=2)
tabla.to_csv(BASE / 'resultados' / f'metricas_{NOMBRE_EXP}.csv')

In [ ]:
# Curvas de entrenamiento y curva Precisión-Recall que genera Ultralytics
for nombre in ['results.png', 'PR_curve.png', 'BoxPR_curve.png']:
    ruta = DIR_EXP / nombre
    if ruta.exists():
        display(IImage(filename=str(ruta), width=900))

## 8. Prueba de rendimiento (FPS)
Mide el tiempo de **preprocesamiento + inferencia** por cuadro. En la Fase 3 se suma el tracking y la lógica de conteo para medir el FPS del sistema completo.
Si subes un video a `VisionArtificial/videos_prueba/`, se usa ese; si no, se usan imágenes de validación.

In [ ]:
import time

videos = sorted((BASE / 'videos_prueba').glob('*.mp4'))
prep = pp.Preprocesador(lado_max=1280, clahe='auto', gamma=True)

if videos:
    print('Video:', videos[0].name, pp.info_video(videos[0]))
    fuente = (f for _, f in pp.iterar_video(videos[0]))
else:
    imgs = sorted((YOLO_LOCAL / 'images' / 'val').glob('*.jpg'))[:300]
    fuente = (cv2.imread(str(p)) for p in imgs)

tiempos = []
for i, frame in enumerate(fuente):
    if i >= 300: break
    t0 = time.perf_counter()
    entrada = prep(frame)
    _ = modelo_ft.predict(entrada, imgsz=CONFIG['imgsz'], conf=0.25, verbose=False, half=True)
    tiempos.append(time.perf_counter() - t0)

t = np.array(tiempos[10:])  # se descartan los primeros cuadros (calentamiento de la GPU)
print(f'Cuadros: {len(t)} | Latencia media: {t.mean()*1000:.1f} ms | FPS: {1/t.mean():.1f}')
print('Preprocesamiento:', prep.resumen_tiempos())

In [ ]:
# Visualización de detecciones sobre un cuadro
res = modelo_ft.predict(entrada, imgsz=CONFIG['imgsz'], conf=0.25, verbose=False)[0]
plt.figure(figsize=(12, 7)); plt.imshow(res.plot()[:, :, ::-1]); plt.axis('off')
plt.title(f'Personas detectadas: {len(res.boxes)}'); plt.show()

## 9. Siguientes pasos (Fase 3: Tracking y lógica de conteo)
- Integrar **ByteTrack** con `modelo_ft.track(..., tracker='bytetrack.yaml', persist=True)` y comparar contra DeepSORT.
- Programar la **línea virtual** con producto cruzado para clasificar entradas y salidas.
- Evaluar el tracking en **MOT20** (carpeta `datasets/mot20/`) con métricas MOTA/IDF1, y el conteo con **MAE** contra conteo manual.